# Notebook 4 ? Bersihkan Isi CSV

Rapikan spasi, samakan penanda nilai kosong, buang baris kosong, dan (opsional) ubah kolom tanggal atau angka. File sumber tetap aman; hasil ditulis sebagai file baru.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI
INPUT_PATH = Path("Input/input.csv")
SEP = ","
NULL_MARKERS = ["", "NA", "N/A", "null", "NULL", "-"]
STRIP_TEXT = True
LOWERCASE_COLUMNS = []  # Contoh: ["email"] ? ubah isi kolom menjadi huruf kecil
DATE_COLUMNS = []       # Contoh: ["tanggal"]
NUMBER_COLUMNS = []     # Contoh: ["harga", "jumlah"]
DROP_FULLY_EMPTY_ROWS = True
OUTPUT_PATH = Path("Output/hasil_clean.csv")
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Terapkan pembersihan

In [ ]:
source = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=SEP, dtype="string", encoding="utf-8-sig", keep_default_na=False)
df.columns = df.columns.astype(str).str.strip()
if STRIP_TEXT:
    for col in df.columns: df[col] = df[col].str.strip()
for col in df.columns:
    df[col] = df[col].replace(NULL_MARKERS, pd.NA)
for col in LOWERCASE_COLUMNS:
    if col not in df: raise KeyError(f"Kolom tidak ditemukan: {col}")
    df[col] = df[col].str.lower()
for col in DATE_COLUMNS:
    if col not in df: raise KeyError(f"Kolom tanggal tidak ditemukan: {col}")
    parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    print(f"{col}: {parsed.isna().sum():,} nilai kosong/tidak terbaca sebagai tanggal")
    df[col] = parsed.dt.strftime("%Y-%m-%d").astype("string")
for col in NUMBER_COLUMNS:
    if col not in df: raise KeyError(f"Kolom angka tidak ditemukan: {col}")
    # Ganti koma desimal menjadi titik sebelum konversi jika format angka Anda memakai koma.
    parsed = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")
    print(f"{col}: {parsed.isna().sum():,} nilai kosong/tidak terbaca sebagai angka")
    df[col] = parsed
before = len(df)
if DROP_FULLY_EMPTY_ROWS: df = df.dropna(how="all")
print(f"Baris kosong dihapus: {before-len(df):,}; sisa: {len(df):,}")
display(df.head())


In [ ]:
output = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False, encoding="utf-8-sig")
print(f"Tersimpan: {output.resolve()}")